# 02 — Pipeline RAG e Exploração da Voz do Cliente (VoC)

Este notebook demonstra o funcionamento completo do pipeline **Retrieval-Augmented Generation (RAG)** desenvolvido para explorar as avaliações de clientes da base Olist:

1. **Transformação da pergunta (*Multi-Query Expansion*):** geração de variações semânticas para ampliar o *recall* sobre termos informais usados pelos consumidores.
2. **Recuperação semântica e seleção:** busca vetorial no `FAISS` com normalização L2/cosseno, deduplicação por `review_id` e filtro de relevância mínima (`RAG_RELEVANCE_THRESHOLD = 0.45`).
3. **Construção do contexto e Geração Fundamentada:** formatação das evidências com metadados e síntese via `ChatOpenAI` (`gpt-4o-mini`) sob *system prompt* estrito de *grounding*.
4. **Tratamento de perguntas sem evidências suficientes:** verificação em dupla camada (*Retrieval Gate* pré-LLM e *Guardrail* pós-LLM) para recusar perguntas fora do escopo sem alucinar.

## 1. Carregamento dos Exemplos Avaliados no Pipeline RAG

In [1]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import EVALUATION_EXAMPLES_PATH

examples = json.loads(EVALUATION_EXAMPLES_PATH.read_text(encoding="utf-8"))
print(f"Total de cenários de exploração carregados: {len(examples)}")
for idx, item in enumerate(examples, start=1):
    tag = "FUNDAMENTADA" if item["has_sufficient_evidence"] else "RECUSA SEGURA"
    print(f"{idx}. [{tag}] {item['question']}")

Total de cenários de exploração carregados: 6
1. [FUNDAMENTADA] Quais são os principais problemas relatados pelos clientes em suas avaliações?
2. [FUNDAMENTADA] Quais reclamações relacionadas à entrega aparecem com maior frequência?
3. [FUNDAMENTADA] O que os clientes mais elogiam em suas experiências de compra?
4. [FUNDAMENTADA] Quais padrões podem ser identificados nas avaliações de clientes insatisfeitos?
5. [FUNDAMENTADA] Quais avaliações sustentam a conclusão de que existem problemas recorrentes relacionados ao prazo de entrega?
6. [RECUSA SEGURA] Qual é a cotação atual do dólar e a previsão do tempo em Tóquio amanhã?


## 2. Inspeção Detalhada: Transformação, Evidências Recuperadas e Resposta Fundamentada

In [2]:
entrega_case = examples[1]
print("=" * 80)
print(f"PERGUNTA: {entrega_case['question']}")
print("=" * 80)
print("\n[Consultas Expandidas (Multi-Query)]:")
for q in entrega_case["expanded_queries"]:
    print(f"  - {q}")

print(f"\n[Evidências Recuperadas ({entrega_case['retrieved_count']})]:")
for idx, ev in enumerate(entrega_case["evidences"][:3], start=1):
    print(f"  {idx}. review_id: {ev['review_id']} | Nota: {ev['review_score']}/5 | Relevância: {ev['relevance_score']:.3f}")
    print(f"     \"{ev['text']}\"")

print("\n[Resposta Gerada pelo LLM]:")
print(entrega_case["answer"])

PERGUNTA: Quais reclamações relacionadas à entrega aparecem com maior frequência?

[Consultas Expandidas (Multi-Query)]:
  - Quais reclamações relacionadas à entrega aparecem com maior frequência?
  - Quais são as reclamações mais comuns sobre entregas?
  - Quais tipos de reclamações sobre entrega são mais frequentes?
  - Quais reclamações sobre entrega ocorrem com mais regularidade?

[Evidências Recuperadas (5)]:
  1. review_id: 72926e5236768ffc912764ca429d0e00 | Nota: 1/5 | Relevância: 0.699
     "Muita demora na entrega"
  2. review_id: 949eca021ed60d00a2d2bda3cc686d4d | Nota: 1/5 | Relevância: 0.654
     "muitos desencontros nas entregas de produtos"
  3. review_id: 7528a396f7625967f05cb09946286608 | Nota: 1/5 | Relevância: 0.648
     "Ultimamente estão atrasando muito as entregas"

[Resposta Gerada pelo LLM]:
As reclamações relacionadas à entrega que aparecem com maior frequência nas avaliações são:

1. **Demora na entrega**: Vários clientes mencionaram a lentidão no processo de e

## 3. Verificação do Tratamento de Perguntas Fora de Escopo / Sem Evidências

In [3]:
refusal_case = examples[-1]
print(f"Pergunta: {refusal_case['question']}")
print(f"Evidências suficientes? {refusal_case['has_sufficient_evidence']}")
print(f"Motivo do bloqueio:     {refusal_case['refusal_reason']}")
print(f"Documentos recuperados: {refusal_case['retrieved_count']}")
print(f"Resposta do sistema:    {refusal_case['answer']}")

Pergunta: Qual é a cotação atual do dólar e a previsão do tempo em Tóquio amanhã?
Evidências suficientes? False
Motivo do bloqueio:     below_relevance_threshold
Documentos recuperados: 0
Resposta do sistema:    Com base nas avaliações disponíveis na base de conhecimento da Olist, não foram encontradas evidências suficientes para responder a esta pergunta com segurança. A questão pode estar fora do escopo dos relatos de clientes ou não possuir ocorrências relevantes registradas.


### Q&A
- **Quais são os principais problemas e reclamações de entrega relatados pelos clientes?** Os consumidores apontam falta de itens entregues no pedido (`review_id: facaaf5d1ea43d6ead5d1288fb8eed21`), atrasos e demora excessiva (`review_id: 72926e5236768ffc912764ca429d0e00` e `7528a396f7625967f05cb09946286608`), desencontros logísticos (`review_id: 949eca021ed60d00a2d2bda3cc686d4d`) e ausência de entrega domiciliar exigindo retirada nos Correios (`review_id: f242ce44c6572a94907a446b8fda32c8`).
- **O que os clientes mais elogiam em suas experiências de compra?** A honestidade e comprometimento da loja (`review_id: 382d855ffdd4f5e66594dbfb1566e96f` e `d7571a8910cfc6319c8acbba887fd33b`), a pontualidade na entrega (`review_id: f8bc084927f01cb84fce083eb9aa5819`) e a excelência no atendimento (`review_id: a644c23f94b9da0ca04ae1de4538e42a`).
- **Como o sistema reage a perguntas sem evidências suficientes?** Quando a consulta não atinge o limiar mínimo de similaridade semântica (`0.45`) ou está fora do domínio da Olist, o pipeline interrompe a geração e retorna uma recusa transparente sem inventar informações.

### Data Analysis Key Findings
- A expansão *Multi-Query* combinada aos embeddings `text-embedding-3-small` recupera avaliações relevantes com similaridade de cosseno entre `0.480` e `0.699` nas 5 investigações de negócio.
- Todas as respostas válidas geradas pelo `gpt-4o-mini` preservam rastreabilidade direta por meio do par `[review_id, Nota]`, permitindo auditar o comentário exato que originou cada conclusão executiva.

### Insights or Next Steps
- Os relatos de insatisfação severa (notas 1 e 2) combinam falhas de prazo e entrega parcial com ausência de comunicação ativa, indicando que alertas proativos de atraso têm alto potencial de redução de reclamações.
- Novas perguntas ad-hoc podem ser executadas diretamente via CLI com `python src/cli.py query -q "<sua pergunta>"`.